In [1]:
from pathlib import Path
import sys

# Resolve paths even if the working directory is repo root.
notebook_dir = Path.cwd()
if not (notebook_dir / "conf.toml").exists():
    notebook_dir = notebook_dir / "notebook"
repo_root = notebook_dir.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

paths = {
    "conf": notebook_dir / "conf.toml",
    "doc": notebook_dir / "blob_documentation.txt",
    "code": notebook_dir / "blob.js",
    "spec": notebook_dir / "blob.webidl",
}

# Circinus demo overview
This notebook runs a small, end-to-end demo of the generation-based fuzzer. It sets up paths, loads config, creates an `Agent`, then fuzzes either the bundled WebIDL artifacts or the small demo files below.

**Note:** This notebook now uses OpenRouter-backed live model calls, so set `OPENROUTER_API_KEY` before running cells that invoke the LLM.

In [2]:
from circinus.settings import load_config

In [3]:
from circinus.agent import Agent

In [4]:
config = load_config(str(paths["conf"]))

In [5]:
agent = Agent(config)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## Telemetry warnings

That means ChromaDB's analytics client is trying to send usage data but the telemetry library signature does not match. It does **not** affect the demo or the fuzzing results. You can ignore these warnings or disable telemetry if you want a clean output.

In [6]:
snippets = agent.fuzz(
    documentation=str(paths["doc"]),
    code=str(paths["code"]),
    specification=str(paths["spec"]),
)

In [ ]:
# Verify OpenRouter config
from circinus.settings import settings
print(f"OpenRouter model: {settings.openrouter_api_model}")
print(f"Base URL: {settings.openrouter_base_url}")
print(f"Config: {settings.to_dict()}")

Mode: mock
Config: {'OPENAI_API_KEY': '${OPENAI_API_KEY}', 'OPENAI_API_MODEL': 'gpt-3.5-turbo', 'MAX_TOKENS': 2048, 'MODE': 'mock'}


## Demo flow
1. Build three small inputs: documentation, specification, and a Python code file.
2. Call `agent.fuzz(...)` to generate candidate prompts and fuzzed snippets.
3. Print a short preview to verify output with a live OpenRouter model.

The same flow now always uses OpenRouter-compatible calls for generation and retrieval components.

In [8]:
from pathlib import Path

# Create simple demo files
demo_dir = Path("demo_files")
demo_dir.mkdir(exist_ok=True)

# Demo Python code
(demo_dir / "demo.py").write_text("""
def add(a, b):
    return a + b

def multiply(a, b):
    return a * b
""")

# Demo documentation
(demo_dir / "doc.txt").write_text("Simple math functions: add and multiply")

# Demo specification
(demo_dir / "spec.txt").write_text("Functions should handle edge cases")

# Now fuzz the demo files
snippets = agent.fuzz(
    documentation=str(demo_dir / "doc.txt"),
    code=str(demo_dir / "demo.py"),
    specification=str(demo_dir / "spec.txt"),
)

# Print first 5 snippets
for i, snippet in enumerate(snippets):
    if i >= 5:
        break
    print(f"--- Snippet {i+1} ---")
    print(snippet)

--- Snippet 1 ---
def clamp(value, low, high):
    return max(low, min(high, value))
--- Snippet 2 ---
def is_even(n):
    return n % 2 == 0
--- Snippet 3 ---
def add_checked(a, b):
    if a is None or b is None:
        return 0
    return a + b
--- Snippet 4 ---
def multiply_or_zero(a, b):
    return 0 if a == 0 or b == 0 else a * b
--- Snippet 5 ---
def safe_divide(a, b):
    if b == 0:
        return None
    return a / b


In [9]:
from itertools import islice

# Status check: generate one snippet to confirm output
snippets_preview = list(islice(agent.fuzz(
    documentation=str(demo_dir / "doc.txt"),
    code=str(demo_dir / "demo.py"),
    specification=str(demo_dir / "spec.txt"),
), 1))

print(f"Preview count: {len(snippets_preview)}")
if snippets_preview:
    print(snippets_preview[0])

Preview count: 1
def safe_divide(a, b):
    if b == 0:
        return None
    return a / b


In [10]:
import re

# Simple heuristic scan for likely risk patterns
RISK_PATTERNS = {
    "division": re.compile(r"/\s*\w+"),
    "none_check": re.compile(r"\bis None\b"),
    "zero_check": re.compile(r"==\s*0"),
    "mutates_state": re.compile(r"\bappend\(|\bpop\(|\bupdate\("),
}


def analyze_snippet(code: str) -> list[str]:
    issues = []
    for label, pattern in RISK_PATTERNS.items():
        if pattern.search(code):
            issues.append(label)
    return issues


print("\nPotential risk signals per snippet:")
for i, snippet in enumerate(snippets_preview or []):
    flags = analyze_snippet(snippet)
    print(f"Snippet {i + 1}: {', '.join(flags) if flags else 'no obvious flags'}")


Potential risk signals per snippet:
Snippet 1: division, zero_check


In [11]:
for snippet in snippets:
    print(snippet)

def add_checked(a, b):
    if a is None or b is None:
        return 0
    return a + b
def multiply_or_zero(a, b):
    return 0 if a == 0 or b == 0 else a * b
def safe_divide(a, b):
    if b == 0:
        return None
    return a / b
